# Prediction Pipeline — Gold V2 (LightGBM)

**Overview**
This notebook applies trained models to generate predictions using the prepared feature set.

It produces station-level demand estimates and derives net flow as a measure of imbalance.

**Objective**
Generate predictions for departures and arrivals and compute net flow for operational analysis.

**Inputs**
- Feature dataset (serving features)
- Trained model artifacts

**Net Flow Calculation**

Predicted departures and arrivals are combined to derive net flow:

net_flow = arrivals - departures

This metric represents station imbalance and is used as the primary output of the system.

In [0]:
%pip install lightgbm
%restart_python

**Model Inference**

This section loads trained models and applies them to the feature dataset to generate predictions for:

- Departures
- Arrivals

Predictions are computed at station-hour level.

## Train DEP + ARR from GOLD_V2 (integrated dataset) - LightGBM FINAL

In [0]:
# ============================================================
# Train DEP + ARR from GOLD_V2 (integrated dataset) - LightGBM
# Step A: Build temporal features (lags + rolling) and store once
# Step B: Progressive monthly train/eval using Hash Compact encoding
# Serverless-safe: no persist/cache, 1 toPandas per month (cache)
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

import pandas as pd
import numpy as np
import zlib

import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# 0) PATHS
# ------------------------------------------------------------
GOLD_V2_DIR  = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/gold/gold_v2_spatiotemporal_events"
FEATURES_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/features/goldv2_features_v1"

EVAL_DIR_DEP = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_dep_lgbm_hashcompact_v1"
EVAL_DIR_ARR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_arr_lgbm_hashcompact_v1"

REBUILD_FEATURES = False  # Set True once to build FEATURES_DIR

# ------------------------------------------------------------
# 1) CONFIG
# ------------------------------------------------------------
# Downtown bounding box
DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX = 43.63, 43.67
DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX = -79.41, -79.37

TRAIN_LOOKBACK_DAYS = 90

HASH_BUCKETS = 512
BUCKET_COL   = "station_bucket"

# Grid over num_leaves (LightGBM's primary complexity param, analogous to n_estimators grid)
NUM_LEAVES_GRID = [63, 127, 255]

base_lgbm_params = dict(
    boosting_type="gbdt",
    num_leaves=127,           # overwritten by grid
    max_depth=-1,             # uncapped; num_leaves controls complexity
    learning_rate=0.05,       # same as XGB
    n_estimators=600,         # same as XGB default
    subsample=0.8,            # same as XGB subsample
    colsample_bytree=0.8,     # same as XGB colsample_bytree
    reg_lambda=1.0,           # same as XGB reg_lambda
    min_child_samples=20,     # LightGBM leaf regularization
    objective="regression",
    n_jobs=-1,
    random_state=42,
    verbose=-1                # suppress LightGBM training logs
)

# Evaluate full period Oct-2022 to Sep-2024
START_Y, START_M = 2022, 10
END_Y,   END_M   = 2024,  9

# ------------------------------------------------------------
# 2) HELPERS
# ------------------------------------------------------------
def path_exists(path: str) -> bool:
    try:
        _ = dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def station_to_bucket(station_id: str, n_buckets: int) -> int:
    if station_id is None:
        return 0
    return zlib.crc32(str(station_id).encode("utf-8")) % n_buckets

def prev_months_in_lookback(test_start: pd.Timestamp, lookback_days: int):
    start = (test_start - pd.Timedelta(days=lookback_days)).to_period("M")
    end   = (test_start - pd.Timedelta(days=1)).to_period("M")
    return [(int(p.year), int(p.month)) for p in pd.period_range(start, end, freq="M")]

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def in_range(y, m, sy, sm, ey, em) -> bool:
    return (y > sy or (y == sy and m >= sm)) and (y < ey or (y == ey and m <= em))

def train_best_model(train_X, train_y, test_X, test_y, leaves_grid, sample_weight=None):
    best_rmse_val = np.inf
    best_mae_val  = None
    best_pred     = None
    best_leaves   = None

    for nl in leaves_grid:
        params = dict(base_lgbm_params)
        params["num_leaves"] = int(nl)

        model = lgb.LGBMRegressor(**params)
        if sample_weight is not None:
            model.fit(train_X, train_y, sample_weight=sample_weight)
        else:
            model.fit(train_X, train_y)

        pred     = model.predict(test_X).astype(np.float32)
        cur_rmse = rmse(test_y, pred)

        if cur_rmse < best_rmse_val:
            best_rmse_val = cur_rmse
            best_mae_val  = float(mean_absolute_error(test_y, pred))
            best_pred     = pred
            best_leaves   = int(nl)

    return best_pred, best_mae_val, best_rmse_val, best_leaves

# ------------------------------------------------------------
# 3) BUILD FEATURES FROM GOLD_V2 (lags + rolling) -> FEATURES_DIR
# ------------------------------------------------------------
if REBUILD_FEATURES:
    if not path_exists(GOLD_V2_DIR):
        raise Exception(f"GOLD_V2_DIR not found: {GOLD_V2_DIR}")

    print("REBUILD_FEATURES=True -> building features from:", GOLD_V2_DIR)
    df = spark.read.parquet(GOLD_V2_DIR)

    df = df.filter(
        (F.col("lat").between(DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX)) &
        (F.col("lon").between(DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX))
    )

    df = (
        df
        .withColumn("date",    F.make_date("year", "month", "day"))
        .withColumn("dow_num", F.dayofweek("date"))
    )

    w = Window.partitionBy("station_id").orderBy(F.col("date"), F.col("hour"))

    df = (
        df
        .withColumn("lag1_dep",   F.lag("departures", 1).over(w))
        .withColumn("lag2_dep",   F.lag("departures", 2).over(w))
        .withColumn("lag24_dep",  F.lag("departures", 24).over(w))
        .withColumn("lag168_dep", F.lag("departures", 168).over(w))
        .withColumn("lag1_arr",   F.lag("arrivals", 1).over(w))
        .withColumn("lag2_arr",   F.lag("arrivals", 2).over(w))
        .withColumn("lag24_arr",  F.lag("arrivals", 24).over(w))
        .withColumn("lag168_arr", F.lag("arrivals", 168).over(w))
    )

    roll_w_3h  = w.rowsBetween(-3,  -1)
    roll_w_24h = w.rowsBetween(-24, -1)

    df = (
        df
        .withColumn("roll_mean_3h_dep",  F.avg("departures").over(roll_w_3h))
        .withColumn("roll_std_24h_dep",  F.stddev("departures").over(roll_w_24h))
        .withColumn("roll_mean_3h_arr",  F.avg("arrivals").over(roll_w_3h))
        .withColumn("roll_std_24h_arr",  F.stddev("arrivals").over(roll_w_24h))
    )

    df.dropna().write.mode("overwrite").partitionBy("year", "month").parquet(FEATURES_DIR)
    print("Features written to:", FEATURES_DIR)

# ------------------------------------------------------------
# 4) LOAD FEATURES
# ------------------------------------------------------------
if not path_exists(FEATURES_DIR):
    raise Exception(
        f"FEATURES_DIR not found: {FEATURES_DIR}\n"
        "Set REBUILD_FEATURES=True once to build features."
    )

df_feat = spark.read.parquet(FEATURES_DIR)
print("Loaded feature rows:", f"{df_feat.count():,}")

# ------------------------------------------------------------
# 5) Column definitions
# ------------------------------------------------------------
weather_cols = ["temperature_2m_celsius", "apparent_temperature_celsius"]

event_numeric_cols = [
    "event_day_flag", "event_day_attendance_sum", "events_day_count",
    "event_active_nearby_flag", "events_nearby_count", "nearest_event_km",
    "event_weighted_intensity", "event_attendance_est_sum_nearby", "event_impact_score",
]

base_cols = ["station_id", "year", "month", "day", "hour", "date", "dow_num"]
DEP_TGT   = "departures"
ARR_TGT   = "arrivals"

dep_hist = ["lag1_dep","lag2_dep","lag24_dep","lag168_dep","roll_mean_3h_dep","roll_std_24h_dep"]
arr_hist = ["lag1_arr","lag2_arr","lag24_arr","lag168_arr","roll_mean_3h_arr","roll_std_24h_arr"]

need_cols = set(base_cols + weather_cols + event_numeric_cols + dep_hist + arr_hist + [DEP_TGT, ARR_TGT])
missing   = [c for c in need_cols if c not in df_feat.columns]
if missing:
    raise Exception(f"Feature dataset missing columns: {missing}")
print("Column check passed")

common_feats = [
    BUCKET_COL, "month", "hour", "dow_num", "is_weekend",
    "temperature_2m_celsius", "apparent_temperature_celsius",
    "event_day_flag", "event_day_attendance_sum", "events_day_count",
    "event_active_nearby_flag", "events_nearby_count", "nearest_event_km",
    "event_weighted_intensity", "event_attendance_est_sum_nearby", "event_impact_score",
]

dep_model_features = common_feats + dep_hist
arr_model_features = common_feats + arr_hist

# ------------------------------------------------------------
# 6) Month list
# ------------------------------------------------------------
months_rows = df_feat.select("year", "month").distinct().collect()
months_list = sorted([(int(r["year"]), int(r["month"])) for r in months_rows])
months_list = [(y,m) for y,m in months_list if in_range(y,m, START_Y,START_M, END_Y,END_M)]
print("Months to evaluate:", len(months_list), "|", months_list[0], "->", months_list[-1])

# ------------------------------------------------------------
# 7) Pandas month cache
# ------------------------------------------------------------
month_cache = {}

select_cols_for_pandas = (
    base_cols + weather_cols + event_numeric_cols + dep_hist + arr_hist + [DEP_TGT, ARR_TGT]
)

def load_month_pd(y: int, m: int) -> pd.DataFrame:
    key = (y, m)
    if key in month_cache:
        return month_cache[key]
    pdf = (
        df_feat
        .filter((F.col("year")==y) & (F.col("month")==m))
        .select(select_cols_for_pandas)
        .toPandas()
    )
    if len(pdf) > 0:
        pdf["date"]      = pd.to_datetime(pdf["date"])
        pdf[BUCKET_COL]  = pdf["station_id"].map(lambda s: station_to_bucket(s, HASH_BUCKETS)).astype(np.int16)
        pdf["is_weekend"]= pdf["dow_num"].isin([1,7]).astype(np.int8)
        for c in weather_cols + event_numeric_cols + dep_hist + arr_hist:
            if c in pdf.columns:
                pdf[c] = pdf[c].fillna(0)
    month_cache[key] = pdf
    return pdf

def build_xy(pdf: pd.DataFrame, features: list, target: str):
    missing = [c for c in features + [target] if c not in pdf.columns]
    if missing:
        raise KeyError(f"Missing cols for target='{target}': {missing}")
    return pdf[features].astype(np.float32).values, pdf[target].astype(np.float32).values

# ------------------------------------------------------------
# 8) Progressive monthly training/eval (DEP + ARR)
# ------------------------------------------------------------
results_dep = []
results_arr = []

for (y, m) in months_list:
    test_start  = pd.Timestamp(year=y, month=m, day=1)
    test_end    = test_start + pd.offsets.MonthEnd(0)
    train_end   = test_start - pd.Timedelta(days=1)
    train_start = train_end  - pd.Timedelta(days=TRAIN_LOOKBACK_DAYS)

    test_pd = load_month_pd(y, m)
    if test_pd.empty:
        continue

    train_parts = []
    for (yy, mm) in prev_months_in_lookback(test_start, TRAIN_LOOKBACK_DAYS):
        if not in_range(yy, mm, START_Y, START_M, END_Y, END_M):
            continue
        part = load_month_pd(yy, mm)
        if not part.empty:
            train_parts.append(part)
    if not train_parts:
        continue

    train_all = pd.concat(train_parts, ignore_index=True)
    train_pd  = train_all[(train_all["date"] >= train_start) & (train_all["date"] <= train_end)]
    test_pd_f = test_pd[(test_pd["date"] >= test_start) & (test_pd["date"] <= test_end)]
    if train_pd.empty or test_pd_f.empty:
        continue

    join_keys = ["station_id", "date", "hour"]
    train_pd  = train_pd.sort_values(join_keys).reset_index(drop=True)
    test_pd_f = test_pd_f.sort_values(join_keys).reset_index(drop=True)

    # ---- DEP ----
    dep_train_X, dep_train_y = build_xy(train_pd, dep_model_features, DEP_TGT)
    dep_test_X,  dep_test_y  = build_xy(test_pd_f, dep_model_features, DEP_TGT)

    dep_baseline     = test_pd_f["lag1_dep"].astype(np.float32).values
    dep_baseline_mae = float(mean_absolute_error(dep_test_y, dep_baseline))
    dep_baseline_rmse= rmse(dep_test_y, dep_baseline)

    dep_pred, dep_mae, dep_rmse_val, dep_best_leaves = train_best_model(
        dep_train_X, dep_train_y, dep_test_X, dep_test_y, NUM_LEAVES_GRID
    )
    dep_impr = (dep_baseline_mae - dep_mae) / dep_baseline_mae * 100 if dep_baseline_mae else np.nan

    results_dep.append({
        "year": y, "month": m, "rows_test": int(len(dep_test_y)),
        "baseline_mae": dep_baseline_mae, "model_mae": float(dep_mae),
        "baseline_rmse": dep_baseline_rmse, "model_rmse": float(dep_rmse_val),
        "improvement_pct": float(dep_impr), "best_num_leaves": int(dep_best_leaves),
        "hash_buckets": HASH_BUCKETS, "train_lookback_days": TRAIN_LOOKBACK_DAYS
    })

    # ---- ARR ----
    arr_train_X, arr_train_y = build_xy(train_pd, arr_model_features, ARR_TGT)
    arr_test_X,  arr_test_y  = build_xy(test_pd_f, arr_model_features, ARR_TGT)

    arr_baseline     = test_pd_f["lag1_arr"].astype(np.float32).values
    arr_baseline_mae = float(mean_absolute_error(arr_test_y, arr_baseline))
    arr_baseline_rmse= rmse(arr_test_y, arr_baseline)

    arr_pred, arr_mae, arr_rmse_val, arr_best_leaves = train_best_model(
        arr_train_X, arr_train_y, arr_test_X, arr_test_y, NUM_LEAVES_GRID
    )
    arr_impr = (arr_baseline_mae - arr_mae) / arr_baseline_mae * 100 if arr_baseline_mae else np.nan

    results_arr.append({
        "year": y, "month": m, "rows_test": int(len(arr_test_y)),
        "baseline_mae": arr_baseline_mae, "model_mae": float(arr_mae),
        "baseline_rmse": arr_baseline_rmse, "model_rmse": float(arr_rmse_val),
        "improvement_pct": float(arr_impr), "best_num_leaves": int(arr_best_leaves),
        "hash_buckets": HASH_BUCKETS, "train_lookback_days": TRAIN_LOOKBACK_DAYS
    })

dep_pd = pd.DataFrame(results_dep).sort_values(["year","month"])
arr_pd = pd.DataFrame(results_arr).sort_values(["year","month"])

print("DEP months evaluated:", len(dep_pd))
print("ARR months evaluated:", len(arr_pd))
display(dep_pd)
display(arr_pd)

spark.createDataFrame(dep_pd).write.mode("overwrite").parquet(EVAL_DIR_DEP)
spark.createDataFrame(arr_pd).write.mode("overwrite").parquet(EVAL_DIR_ARR)
print("Saved DEP eval to:", EVAL_DIR_DEP)
print("Saved ARR eval to:", EVAL_DIR_ARR)

## WEIGHT = 10 con JSON - FINAL (LightGBM)

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import zlib, json
from datetime import datetime, timezone

import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============================================================
# Net Flow (Derived) - GOLD_V2 Features (Weighted Training)
# LightGBM version
#
# Serving (Serverless-safe):
# - Train FINAL dep/arr models on ALL available data
# - Save models as LightGBM text format to DBFS via atomic writes
# - Save feature list metadata (JSON) for safe scoring
# - Validate artifacts after write
# ============================================================

# ------------------------------------------------------------
# 0) Paths
# ------------------------------------------------------------
FEATURES_GOLDV2_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/features/goldv2_features_v1"

USE_BEST_N_FROM_PREV_RUNS = False
EVAL_DEP_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_dep_lgbm_hashcompact_v1"
EVAL_ARR_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_arr_lgbm_hashcompact_v1"

WEIGHT_EVENT   = 10
EVENT_FLAG_COL = "event_active_nearby_flag"

EVAL_DIR_NETFLOW = (
    f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/"
    f"goldv2_netflow_derived_lgbm_hashcompact_v_final_weight{WEIGHT_EVENT}"
)

# LightGBM models saved as text (human-readable, no binary encoding needed)
SERVING_MODEL_DIR_DBFS = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models_lgbm"
DEP_MODEL_TXT_DBFS     = f"{SERVING_MODEL_DIR_DBFS}/dep_lgbm_weight{WEIGHT_EVENT}.txt"
ARR_MODEL_TXT_DBFS     = f"{SERVING_MODEL_DIR_DBFS}/arr_lgbm_weight{WEIGHT_EVENT}.txt"

SERVING_META_DIR_DBFS  = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata_lgbm"
FEATURE_META_DIR_DBFS  = f"{SERVING_META_DIR_DBFS}/features_weight{WEIGHT_EVENT}"
FEATURE_META_JSON_DBFS = f"{FEATURE_META_DIR_DBFS}/features.json"

MAX_MODEL_BYTES = 50_000_000  # LightGBM text models are more compact than XGB JSON
MAX_META_BYTES  = 500_000

# ------------------------------------------------------------
# 1) Config
# ------------------------------------------------------------
TRAIN_LOOKBACK_DAYS = 90
HASH_BUCKETS        = 512
BUCKET_COL          = "station_bucket"

NUM_LEAVES_GRID = [63, 127, 255]

base_lgbm_params = dict(
    boosting_type="gbdt",
    num_leaves=127,
    max_depth=-1,
    learning_rate=0.05,
    n_estimators=600,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_samples=20,
    objective="regression",
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

# ------------------------------------------------------------
# 2) Helpers
# ------------------------------------------------------------
def path_exists(path: str) -> bool:
    try:
        _ = dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def ensure_dir_dbfs(dir_path: str):
    dbutils.fs.mkdirs(dir_path)

def atomic_put_text(dbfs_path: str, text: str):
    dst_dir = dbfs_path.rsplit("/", 1)[0]
    ensure_dir_dbfs(dst_dir)
    tmp = dbfs_path + ".tmp"
    dbutils.fs.put(tmp, text, True)
    try:
        dbutils.fs.rm(dbfs_path, True)
    except Exception:
        pass
    dbutils.fs.mv(tmp, dbfs_path, True)

def read_dbfs_text(dbfs_path: str, max_bytes: int) -> str:
    return dbutils.fs.head(dbfs_path, max_bytes)

def station_to_bucket(station_id: str, n_buckets: int) -> int:
    if station_id is None:
        return 0
    return zlib.crc32(str(station_id).encode("utf-8")) % n_buckets

def prev_months_in_lookback(test_start: pd.Timestamp, lookback_days: int):
    start = (test_start - pd.Timedelta(days=lookback_days)).to_period("M")
    end   = (test_start - pd.Timedelta(days=1)).to_period("M")
    return [(int(p.year), int(p.month)) for p in pd.period_range(start, end, freq="M")]

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def build_xy(pdf: pd.DataFrame, features: list, target: str):
    missing = [c for c in features + [target] if c not in pdf.columns]
    if missing:
        raise KeyError(f"Missing columns for target='{target}': {missing}")
    return pdf[features].astype(np.float32).values, pdf[target].astype(np.float32).values

def build_sample_weights(pdf: pd.DataFrame, weight_event: float, event_col: str):
    w = np.ones(len(pdf), dtype=np.float32)
    if event_col in pdf.columns and weight_event is not None and weight_event > 1:
        ev = pd.to_numeric(pdf[event_col], errors="coerce").fillna(0).astype(np.int8).values
        w[ev == 1] = float(weight_event)
    return w

def train_best_model(train_X, train_y, test_X, test_y, leaves_grid, forced_best_leaves=None, sample_weight=None):
    candidates = [forced_best_leaves] if (forced_best_leaves is not None and forced_best_leaves > 0) else leaves_grid

    best_rmse_val = np.inf
    best_mae_val  = None
    best_pred     = None
    best_leaves   = None
    best_model    = None

    for nl in candidates:
        params = dict(base_lgbm_params)
        params["num_leaves"] = int(nl)

        model = lgb.LGBMRegressor(**params)
        if sample_weight is not None:
            model.fit(train_X, train_y, sample_weight=sample_weight)
        else:
            model.fit(train_X, train_y)

        pred     = model.predict(test_X).astype(np.float32)
        cur_rmse = rmse(test_y, pred)

        if cur_rmse < best_rmse_val:
            best_rmse_val = cur_rmse
            best_mae_val  = float(mean_absolute_error(test_y, pred))
            best_pred     = pred
            best_leaves   = int(nl)
            best_model    = model

    return best_model, best_pred, best_mae_val, best_rmse_val, best_leaves

def save_lgbm_model_txt_dbfs(model: lgb.LGBMRegressor, dbfs_path: str):
    """
    Serverless-safe:
    - Export LightGBM booster as text (native LightGBM format)
    - Atomic write using tmp -> mv pattern
    """
    model_str = model.booster_.model_to_string()
    if len(model_str) < 1000:
        raise Exception(f"Model text suspiciously small before write: {dbfs_path}")
    atomic_put_text(dbfs_path, model_str)
    print(f"Saved LightGBM model to: {dbfs_path} (bytes={len(model_str)})")

def load_lgbm_model_from_dbfs(dbfs_path: str, max_bytes: int = MAX_MODEL_BYTES) -> lgb.Booster:
    model_str = read_dbfs_text(dbfs_path, max_bytes)
    if len(model_str) < 1000:
        raise Exception(f"Model text looks too small (possible truncation): {dbfs_path}")
    return lgb.Booster(model_str=model_str)

def save_json_dbfs(obj: dict, dbfs_path: str):
    json_str = json.dumps(obj, indent=2)
    if len(json_str) < 50:
        raise Exception(f"Metadata JSON suspiciously small: {dbfs_path}")
    atomic_put_text(dbfs_path, json_str)
    print(f"Saved metadata JSON to: {dbfs_path}")

def validate_artifacts(dep_path: str, arr_path: str, meta_path: str, expected_n_estimators: int):
    for p in [dep_path, arr_path, meta_path]:
        if not path_exists(p):
            raise Exception(f"Validation failed: missing artifact {p}")

    meta     = json.loads(read_dbfs_text(meta_path, max_bytes=MAX_META_BYTES))
    dep_feats= meta.get("dep_features", [])
    arr_feats= meta.get("arr_features", [])
    hb       = int(meta.get("hash_buckets", -1))
    if not dep_feats or not arr_feats:
        raise Exception("Validation failed: metadata missing dep_features/arr_features")
    if hb <= 0:
        raise Exception("Validation failed: hash_buckets invalid")

    dep_booster = load_lgbm_model_from_dbfs(dep_path)
    arr_booster = load_lgbm_model_from_dbfs(arr_path)

    # Smoke predict
    X_dep = np.zeros((2, len(dep_feats)), dtype=np.float32)
    X_arr = np.zeros((2, len(arr_feats)), dtype=np.float32)
    dep_smoke = dep_booster.predict(X_dep)
    arr_smoke = arr_booster.predict(X_arr)

    if np.isnan(dep_smoke).any() or np.isnan(arr_smoke).any():
        raise Exception("NaNs in smoke predictions during validation")

    print(f"Validation OK: dep_features={len(dep_feats)}, arr_features={len(arr_feats)}, hash_buckets={hb}")
    print(f"Boosters loaded + smoke predict OK")
    print("ALL ARTIFACT VALIDATIONS PASSED")

# ------------------------------------------------------------
# 3) Load Features
# ------------------------------------------------------------
if not path_exists(FEATURES_GOLDV2_DIR):
    raise Exception(f"FEATURES_GOLDV2_DIR not found: {FEATURES_GOLDV2_DIR}")

df_feat = spark.read.parquet(FEATURES_GOLDV2_DIR)

dep_target = "departures"
arr_target = "arrivals"

# ------------------------------------------------------------
# 4) Column definitions
# ------------------------------------------------------------
key_cols      = ["station_id", "year", "month", "day", "hour", "date", "dow_num"]
dep_lag_cols  = ["lag1_dep", "lag2_dep", "lag24_dep", "lag168_dep"]
arr_lag_cols  = ["lag1_arr", "lag2_arr", "lag24_arr", "lag168_arr"]
dep_roll_cols = ["roll_mean_3h_dep", "roll_std_24h_dep"]
arr_roll_cols = ["roll_mean_3h_arr", "roll_std_24h_arr"]
weather_cols  = ["temperature_2m_celsius", "apparent_temperature_celsius"]
event_cols    = [
    "event_day_flag", "events_day_count", "event_day_attendance_sum",
    EVENT_FLAG_COL, "events_nearby_count", "nearest_event_km",
    "event_weighted_intensity", "event_attendance_est_sum_nearby", "event_impact_score"
]

common_features    = [BUCKET_COL, "month", "hour", "dow_num", "is_weekend", *weather_cols, *event_cols]
dep_model_features = list(common_features + dep_lag_cols + dep_roll_cols)
arr_model_features = list(common_features + arr_lag_cols + arr_roll_cols)

# Fail fast
cols_set  = set(df_feat.columns)
must_have = (
    key_cols + [dep_target, arr_target] +
    dep_lag_cols + arr_lag_cols + dep_roll_cols + arr_roll_cols +
    weather_cols + event_cols
)
missing = [c for c in must_have if c not in cols_set]
if missing:
    raise Exception(f"GOLDV2 features dataset missing columns: {missing}")
print("Column check passed")

# ------------------------------------------------------------
# 5) Month list
# ------------------------------------------------------------
months_list = [(int(r["year"]), int(r["month"])) for r in (
    df_feat.select("year", "month").distinct().orderBy("year", "month").collect()
)]
print("Months in dataset:", len(months_list))

# ------------------------------------------------------------
# 6) Optional: Load best_leaves per month from prev runs
# ------------------------------------------------------------
best_leaves_dep = {}
best_leaves_arr = {}

if USE_BEST_N_FROM_PREV_RUNS:
    dep_prev = spark.read.parquet(EVAL_DEP_DIR).toPandas()
    arr_prev = spark.read.parquet(EVAL_ARR_DIR).toPandas()
    for _, r in dep_prev.iterrows():
        best_leaves_dep[(int(r["year"]), int(r["month"]))] = int(r.get("best_num_leaves", -1))
    for _, r in arr_prev.iterrows():
        best_leaves_arr[(int(r["year"]), int(r["month"]))] = int(r.get("best_num_leaves", -1))
    print("Loaded per-month best_num_leaves from previous runs")

# ------------------------------------------------------------
# 7) Pandas month cache
# ------------------------------------------------------------
month_cache = {}
numeric_cols = weather_cols + event_cols + dep_lag_cols + arr_lag_cols + dep_roll_cols + arr_roll_cols

month_select_cols = (
    key_cols + [dep_target, arr_target] +
    dep_lag_cols + arr_lag_cols + dep_roll_cols + arr_roll_cols +
    weather_cols + event_cols
)

def load_month(y: int, m: int) -> pd.DataFrame:
    key = (y, m)
    if key in month_cache:
        return month_cache[key]
    pdf = (
        df_feat
        .filter((F.col("year") == y) & (F.col("month") == m))
        .select(month_select_cols)
        .toPandas()
    )
    if len(pdf) > 0:
        pdf["date"]      = pd.to_datetime(pdf["date"])
        pdf[BUCKET_COL]  = pdf["station_id"].map(lambda s: station_to_bucket(s, HASH_BUCKETS)).astype(np.int16)
        pdf["is_weekend"]= pdf["dow_num"].isin([1, 7]).astype(np.int8)
        for c in numeric_cols:
            if c in pdf.columns:
                pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(0.0)
    month_cache[key] = pdf
    return pdf

# ------------------------------------------------------------
# 8) Monthly loop: train dep + arr, derive net_flow, evaluate
# ------------------------------------------------------------
results = []
skipped_no_train = 0

for (y, m) in months_list:
    test_start  = pd.Timestamp(year=y, month=m, day=1)
    test_end    = test_start + pd.offsets.MonthEnd(0)
    train_end   = test_start - pd.Timedelta(days=1)
    train_start = train_end  - pd.Timedelta(days=TRAIN_LOOKBACK_DAYS)

    test_pd = load_month(y, m)
    if test_pd.empty:
        continue

    train_parts = []
    for (yy, mm) in prev_months_in_lookback(test_start, TRAIN_LOOKBACK_DAYS):
        part = load_month(yy, mm)
        if not part.empty:
            train_parts.append(part)
    if not train_parts:
        skipped_no_train += 1
        continue

    train_all = pd.concat(train_parts, ignore_index=True)
    train_pd  = train_all[(train_all["date"] >= train_start) & (train_all["date"] <= train_end)]
    test_pd_f = test_pd[(test_pd["date"] >= test_start) & (test_pd["date"] <= test_end)]
    if train_pd.empty or test_pd_f.empty:
        skipped_no_train += 1
        continue

    join_keys = ["station_id", "date", "hour"]
    train_pd  = train_pd.sort_values(join_keys).reset_index(drop=True)
    test_pd_f = test_pd_f.sort_values(join_keys).reset_index(drop=True)

    dep_train_X, dep_train_y = build_xy(train_pd, dep_model_features, dep_target)
    dep_test_X,  dep_test_y  = build_xy(test_pd_f, dep_model_features, dep_target)
    arr_train_X, arr_train_y = build_xy(train_pd, arr_model_features, arr_target)
    arr_test_X,  arr_test_y  = build_xy(test_pd_f, arr_model_features, arr_target)

    train_w = build_sample_weights(train_pd, WEIGHT_EVENT, EVENT_FLAG_COL)

    forced_dep = best_leaves_dep.get((y,m), None) if USE_BEST_N_FROM_PREV_RUNS else None
    forced_arr = best_leaves_arr.get((y,m), None) if USE_BEST_N_FROM_PREV_RUNS else None

    _, dep_pred, _, _, dep_best = train_best_model(
        dep_train_X, dep_train_y, dep_test_X, dep_test_y,
        NUM_LEAVES_GRID, forced_best_leaves=forced_dep, sample_weight=train_w
    )
    _, arr_pred, _, _, arr_best = train_best_model(
        arr_train_X, arr_train_y, arr_test_X, arr_test_y,
        NUM_LEAVES_GRID, forced_best_leaves=forced_arr, sample_weight=train_w
    )

    net_real     = (arr_test_y - dep_test_y).astype(np.float32)
    net_pred     = (arr_pred   - dep_pred).astype(np.float32)
    baseline_net = (
        test_pd_f["lag1_arr"].astype(np.float32).values
        - test_pd_f["lag1_dep"].astype(np.float32).values
    )

    baseline_mae   = float(mean_absolute_error(net_real, baseline_net))
    model_mae      = float(mean_absolute_error(net_real, net_pred))
    improvement_pct= (baseline_mae - model_mae) / baseline_mae * 100 if baseline_mae else np.nan

    event_flag = pd.to_numeric(test_pd_f[EVENT_FLAG_COL], errors="coerce").fillna(0).astype(np.int8).values
    idx_event  = (event_flag == 1)
    idx_noev   = (event_flag == 0)

    seg = {}
    seg["rows_event"]    = int(idx_event.sum())
    seg["rows_no_event"] = int(idx_noev.sum())

    def _seg_metrics(idx, prefix):
        if idx.sum() > 0:
            seg[f"{prefix}_baseline_mae"] = float(mean_absolute_error(net_real[idx], baseline_net[idx]))
            seg[f"{prefix}_model_mae"]    = float(mean_absolute_error(net_real[idx], net_pred[idx]))
            seg[f"{prefix}_baseline_rmse"]= rmse(net_real[idx], baseline_net[idx])
            seg[f"{prefix}_model_rmse"]   = rmse(net_real[idx], net_pred[idx])
        else:
            for k in ["baseline_mae","model_mae","baseline_rmse","model_rmse"]:
                seg[f"{prefix}_{k}"] = np.nan

    _seg_metrics(idx_event, "event")
    _seg_metrics(idx_noev,  "no_event")

    results.append({
        "year": y, "month": m, "rows_test": int(len(net_real)),
        "baseline_mae_net": baseline_mae, "model_mae_net": model_mae,
        "baseline_rmse_net": rmse(net_real, baseline_net), "model_rmse_net": rmse(net_real, net_pred),
        "improvement_pct_net": float(improvement_pct),
        "dep_best_num_leaves": int(dep_best), "arr_best_num_leaves": int(arr_best),
        "hash_buckets": HASH_BUCKETS, "train_lookback_days": TRAIN_LOOKBACK_DAYS,
        "weight_event": float(WEIGHT_EVENT),
        **seg
    })

print(f"Done. results months={len(results)} skipped={skipped_no_train}")

# ------------------------------------------------------------
# 9) Save monthly metrics
# ------------------------------------------------------------
results_pd = pd.DataFrame(results)
display(results_pd)
spark.createDataFrame(results_pd).write.mode("overwrite").parquet(EVAL_DIR_NETFLOW)
print("Saved netflow eval to:", EVAL_DIR_NETFLOW)

# ============================================================
# 10) Train FINAL serving models on ALL data
# ============================================================
print("\n==================== TRAIN FINAL SERVING MODELS ====================")

all_parts = [load_month(yy, mm) for yy, mm in months_list]
all_parts = [p for p in all_parts if not p.empty]
if not all_parts:
    raise Exception("No data to train final models.")

full_pd = pd.concat(all_parts, ignore_index=True).sort_values(["station_id","date","hour"]).reset_index(drop=True)

dep_X, dep_y = build_xy(full_pd, dep_model_features, dep_target)
arr_X, arr_y = build_xy(full_pd, arr_model_features, arr_target)
full_w       = build_sample_weights(full_pd, WEIGHT_EVENT, EVENT_FLAG_COL)

FINAL_NUM_LEAVES = 127

dep_params = dict(base_lgbm_params); dep_params["num_leaves"] = FINAL_NUM_LEAVES
arr_params = dict(base_lgbm_params); arr_params["num_leaves"] = FINAL_NUM_LEAVES

dep_model = lgb.LGBMRegressor(**dep_params)
arr_model = lgb.LGBMRegressor(**arr_params)
dep_model.fit(dep_X, dep_y, sample_weight=full_w)
arr_model.fit(arr_X, arr_y, sample_weight=full_w)

ensure_dir_dbfs(SERVING_MODEL_DIR_DBFS)
save_lgbm_model_txt_dbfs(dep_model, DEP_MODEL_TXT_DBFS)
save_lgbm_model_txt_dbfs(arr_model, ARR_MODEL_TXT_DBFS)

meta = {
    "created_utc":       datetime.now(timezone.utc).isoformat(),
    "model_type":        "LGBMRegressor",
    "weight_event":      WEIGHT_EVENT,
    "hash_buckets":      HASH_BUCKETS,
    "final_num_leaves":  FINAL_NUM_LEAVES,
    "n_estimators":      base_lgbm_params["n_estimators"],
    "event_flag_col":    EVENT_FLAG_COL,
    "dep_features":      dep_model_features,
    "arr_features":      arr_model_features,
    "numeric_cols":      numeric_cols,
}
ensure_dir_dbfs(FEATURE_META_DIR_DBFS)
save_json_dbfs(meta, FEATURE_META_JSON_DBFS)

print("\nFINAL SERVING ARTIFACTS")
print("DEP TXT:", DEP_MODEL_TXT_DBFS)
print("ARR TXT:", ARR_MODEL_TXT_DBFS)
print("META JSON:", FEATURE_META_JSON_DBFS)

print("\n==================== POST-WRITE VALIDATION ====================")
validate_artifacts(
    dep_path=DEP_MODEL_TXT_DBFS,
    arr_path=ARR_MODEL_TXT_DBFS,
    meta_path=FEATURE_META_JSON_DBFS,
    expected_n_estimators=FINAL_NUM_LEAVES
)

## Validacion del modelo (LightGBM)

In [0]:
import json
import numpy as np
import lightgbm as lgb

WEIGHT_EVENT   = 10
DEP_MODEL_DBFS = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models_lgbm/dep_lgbm_weight{WEIGHT_EVENT}.txt"
ARR_MODEL_DBFS = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models_lgbm/arr_lgbm_weight{WEIGHT_EVENT}.txt"
META_DBFS      = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata_lgbm/features_weight{WEIGHT_EVENT}/features.json"

def path_exists(path):
    try: dbutils.fs.ls(path); return True
    except: return False

def read_dbfs_text(p, mb=50_000_000): return dbutils.fs.head(p, mb)
def load_lgbm(p): return lgb.Booster(model_str=read_dbfs_text(p))

print("=== VALIDATE LGBM MODEL ARTIFACTS ===")

for p in [DEP_MODEL_DBFS, ARR_MODEL_DBFS, META_DBFS]:
    if not path_exists(p): raise Exception(f"Missing artifact: {p}")
print("DBFS paths exist")

meta       = json.loads(read_dbfs_text(META_DBFS, 300_000))
dep_feats  = meta["dep_features"]
arr_feats  = meta["arr_features"]
model_type = meta.get("model_type", "?")

assert isinstance(dep_feats, list) and len(dep_feats) > 0, "dep_features missing/empty"
assert isinstance(arr_feats, list) and len(arr_feats) > 0, "arr_features missing/empty"
print(f"Metadata OK. model_type={model_type}, dep={len(dep_feats)} feats, arr={len(arr_feats)} feats")

dep_booster = load_lgbm(DEP_MODEL_DBFS)
arr_booster = load_lgbm(ARR_MODEL_DBFS)
print("Boosters loaded")
print(f"dep num_trees: {dep_booster.num_trees()}, arr num_trees: {arr_booster.num_trees()}")

assert dep_booster.num_trees() > 0 and arr_booster.num_trees() > 0, "Boosters have 0 trees"

X_dep   = np.zeros((2, len(dep_feats)), dtype=np.float32)
X_arr   = np.zeros((2, len(arr_feats)), dtype=np.float32)
dep_pred= dep_booster.predict(X_dep)
arr_pred= arr_booster.predict(X_arr)

assert not np.isnan(dep_pred).any() and not np.isnan(arr_pred).any(), "NaNs in smoke predictions"
print("Smoke prediction OK (no NaNs)")
print("LGBM MODEL ARTIFACTS VALIDATED")

**Results**

The final output includes predicted demand signals and derived net flow values.

These results can be used for monitoring, decision-making, and integration with downstream applications.